# FOXF1_bead — 01_small_large_alignment

**Feeds:** Fig 2c, ED Fig 3d

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# Small-to-Large Alignment Workflow (2026-01-22)

This notebook computes and validates SMALL↔LARGE alignment for all strict filename pairs in:
`data/2026-01-22_PDMS/day2/single/`.

Outputs:
- Transform table: `results/annotations/small_to_large_pair_transforms.tsv`
- Per-pair QC images: `results/qc/01_small_large_alignment/per_pair/*.png`
- QC summary table: `results/qc/01_small_large_alignment/per_pair_summary.tsv`
- Mapped live small bead table: `results/annotations/well_centroids_small_mapped.tsv`
- Inline proof images in notebook output, displayed from the saved QC PNG files.
- Large-image z selection is focus-first: the large BF slice is chosen from the central small-image-sized region so the cyst is in focus for QC and annotation.

Progress is printed during transform computation and QC rendering.


## Imports And Root Resolution


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image as IPyImage, Markdown, display

# Keep matplotlib cache writable inside the workspace.
MPL_TMP = (
    (Path.cwd().resolve() / "../.mplconfig").resolve()
    if (Path.cwd().resolve().name == "notebooks")
    else (Path.cwd().resolve() / ".mplconfig")
)
MPL_TMP.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_TMP))

# Resolve project root whether notebook is launched from repo root or /notebooks.
CWD = Path.cwd().resolve()
if (CWD / "scripts").exists() and (CWD / "results").exists():
    ROOT = CWD
elif (CWD.parent / "scripts").exists() and (CWD.parent / "results").exists():
    ROOT = CWD.parent
else:
    ROOT = CWD

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import manual_well_annotation as mwa
from scripts import centroid_mapping as cmap

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

print("ROOT:", ROOT)
print("Python:", sys.executable)


## Paths And Parameters


In [ ]:
# Parameters
import os

TARGET_SUBDIR = "data/2026-01-22_PDMS/day2/single/"
COHORT_ID = "2026-01-22_day2_live"

PAIR_TRANSFORM_TSV = ROOT / "results/annotations/small_to_large_pair_transforms.tsv"
LIVE_SMALL_MAPPED_CENTROIDS_TSV = ROOT / "results/annotations/well_centroids_small_mapped.tsv"
QC_ROOT = ROOT / "results/qc/01_small_large_alignment"
QC_DIR = QC_ROOT / "per_pair"
MANUAL_REVIEW_QC_DIR = QC_ROOT / "manual_review"
MANUAL_REVIEW_QC_SUMMARY_TSV = QC_ROOT / "manual_review_summary.tsv"

LIMIT = None  # set int for a quick subset
FORCE_RECOMPUTE_TRANSFORMS = False
SHOW_INLINE_QC = os.environ.get("FOXF1_ALIGNMENT_INLINE_QC", "0").lower() not in {"0", "false", "no"}

# Registration settings (rotation allowed, scaling disallowed by implementation).
ALLOW_ROTATION = True
MAX_ABS_ANGLE_DEG = 6.0
COARSE_STEP_DEG = 0.5
FINE_STEP_DEG = 0.1
SEARCH_RADIUS_PX = 420
ALLOW_MISSING_LARGE_FILES = True

MANUAL_SMALL_LARGE_OVERRIDES = {
    "2-6": {
        "anchor_rule": "top_left",
        "search_half_range_deg": 4.0,
        "coarse_step_deg": 0.25,
        "fine_step_deg": 0.05,
        "note": "Manual correction: keep the top-left true well fixed and search only over theta. The lower small-image annotation is a dragged bead, not a well.",
    }
}

print("TARGET_SUBDIR:", TARGET_SUBDIR)
print("COHORT_ID:", COHORT_ID)
print("PAIR_TRANSFORM_TSV:", PAIR_TRANSFORM_TSV)
print("LIVE_SMALL_MAPPED_CENTROIDS_TSV:", LIVE_SMALL_MAPPED_CENTROIDS_TSV)
print("QC_DIR:", QC_DIR)
print("MANUAL_REVIEW_QC_DIR:", MANUAL_REVIEW_QC_DIR)
print("MANUAL_REVIEW_QC_SUMMARY_TSV:", MANUAL_REVIEW_QC_SUMMARY_TSV)
print("LIMIT:", LIMIT)
print("FORCE_RECOMPUTE_TRANSFORMS:", FORCE_RECOMPUTE_TRANSFORMS)
print("SHOW_INLINE_QC (live matplotlib rendering):", SHOW_INLINE_QC)
print("ALLOW_MISSING_LARGE_FILES:", ALLOW_MISSING_LARGE_FILES)
print("MANUAL_SMALL_LARGE_OVERRIDES:", MANUAL_SMALL_LARGE_OVERRIDES)


## Discover Small-Large Pairs


In [ ]:
# Build strict small/large pairs from the 2026-01-22 folder.
pairs = mwa.paired_records_from_directory(
    root=ROOT,
    target_subdir=TARGET_SUBDIR,
    cohort_id=COHORT_ID,
    limit=LIMIT,
    allow_missing_large=ALLOW_MISSING_LARGE_FILES,
)

base = (ROOT / TARGET_SUBDIR).resolve()
small_paths = sorted([pp for pp in base.glob("*.czi") if not pp.name.endswith("-large.czi")])
missing_large_positions = []
for small_abs in small_paths:
    large_abs = small_abs.with_name(f"{small_abs.stem}-large.czi")
    if not large_abs.exists():
        missing_large_positions.append(small_abs.stem)

pairs_df = pd.DataFrame([p.__dict__ for p in pairs])
print(f"Paired records: {len(pairs)}")
if missing_large_positions:
    print("Missing strict large files skipped:", ", ".join(missing_large_positions))
if not pairs_df.empty:
    bad_small = pairs_df["small_file_path"].str.contains("-large.czi", regex=False).sum()
    bad_large = (~pairs_df["large_file_path"].str.endswith("-large.czi")).sum()
    print("Rows with invalid small path suffix:", int(bad_small))
    print("Rows with invalid large path suffix:", int(bad_large))
    if int(bad_small) > 0 or int(bad_large) > 0:
        raise RuntimeError("Pair validation failed: unexpected filename suffixes.")
    display(pairs_df)
else:
    print("No strict pairs found.")


## Compute Automatic Small-Large Transforms


In [ ]:
# Compute/update transforms with progress output.
transforms_df = mwa.compute_pair_transforms(
    pairs=pairs,
    root=ROOT,
    transform_path=PAIR_TRANSFORM_TSV,
    force_recompute=FORCE_RECOMPUTE_TRANSFORMS,
    allow_rotation=ALLOW_ROTATION,
    max_abs_angle_deg=MAX_ABS_ANGLE_DEG,
    coarse_step_deg=COARSE_STEP_DEG,
    fine_step_deg=FINE_STEP_DEG,
    search_radius_px=SEARCH_RADIUS_PX,
    verbose=True,
)

print("Saved transforms:", len(transforms_df))
status_counts = transforms_df["registration_status"].value_counts(dropna=False)
print("Registration status counts:")
display(status_counts.to_frame("n"))

failed = transforms_df[transforms_df["registration_status"] != "ok"].copy()
if not failed.empty:
    print("Non-ok transform rows:", len(failed))
    display(failed[["canonical_position", "small_file_path", "large_file_path", "registration_status"]])

display(transforms_df)


## Review 2-6 Manual Override

`2-6` remains flagged for additional review in this notebook. We keep the global BF template matcher for every scene, then explicitly replace the saved `2-6` geometry with a manual correction that uses the **top-left true well** as a fixed anchor. Translation is recomputed for every tested `theta` so that this anchor remains fixed in large-image coordinates.

Future note: if we ever want to replace the `01` alignment globally with anchor-based manual correction, this same anchored-rotation method could be generalized. That is not necessary for the current dataset.


In [ ]:
# Apply scene-specific manual small-to-large overrides after the automatic pass.
from datetime import datetime

SMALL_CENTROID_TSV = ROOT / "results/annotations/well_centroids.tsv"
LARGE_CENTROID_TSV = ROOT / "results/annotations/well_centroids_large.tsv"

small_centroids_df = mwa.load_centroid_table(SMALL_CENTROID_TSV)
large_centroids_df = mwa.load_large_centroid_table(LARGE_CENTROID_TSV)
override_rows = []

for position, spec in MANUAL_SMALL_LARGE_OVERRIDES.items():
    current_rows = transforms_df[transforms_df["canonical_position"].astype(str) == str(position)].copy()
    if current_rows.empty:
        print(f"Override skipped for {position}: transform row not found.")
        continue

    row = current_rows.iloc[0].to_dict()
    pair_matches = [pp for pp in pairs if str(pp.canonical_position) == str(position)]
    if not pair_matches:
        print(f"Override skipped for {position}: pair not found in active pairs.")
        continue
    pair = pair_matches[0]

    large_loaded = mwa.load_large_registered_stack(pair=pair, root=ROOT)
    small_loaded = mwa.load_primary_image(
        mwa.ImageRecord(
            image_id=pair.image_id,
            cohort_id=pair.cohort_id,
            canonical_position=pair.canonical_position,
            file_path=pair.small_file_path,
        ),
        root=ROOT,
    )
    auto_reg = mwa.register_small_to_large(
        small_bf=small_loaded.bf,
        large_bf_stack=large_loaded.bf_stack,
        allow_rotation=ALLOW_ROTATION,
        max_abs_angle_deg=MAX_ABS_ANGLE_DEG,
        coarse_step_deg=COARSE_STEP_DEG,
        fine_step_deg=FINE_STEP_DEG,
        search_radius_px=SEARCH_RADIUS_PX,
    )

    small_sub = small_centroids_df[
        (small_centroids_df["cohort_id"].astype(str) == str(COHORT_ID))
        & (small_centroids_df["canonical_position"].astype(str) == str(position))
    ].copy().sort_values("centroid_index")
    large_sub = large_centroids_df[
        (large_centroids_df["cohort_id"].astype(str) == str(COHORT_ID))
        & (large_centroids_df["canonical_position"].astype(str) == str(position))
    ].copy().sort_values("centroid_index")

    small_anchor_row = mwa.select_centroid_by_spatial_rule(
        small_sub,
        rule=str(spec.get("anchor_rule", "top_left")),
    )
    large_anchor_row = mwa.select_centroid_by_spatial_rule(
        large_sub,
        rule=str(spec.get("anchor_rule", "top_left")),
    )
    anchored = mwa.register_small_to_large_with_fixed_anchor_rotation(
        small_bf=small_loaded.bf,
        large_bf_stack=large_loaded.bf_stack,
        small_anchor_xy=(float(small_anchor_row["centroid_x_px"]), float(small_anchor_row["centroid_y_px"])),
        large_anchor_xy=(float(large_anchor_row["centroid_x_px"]), float(large_anchor_row["centroid_y_px"])),
        initial_theta_deg=float(auto_reg["small_to_large_theta_deg"]),
        search_half_range_deg=float(spec.get("search_half_range_deg", 4.0)),
        coarse_step_deg=float(spec.get("coarse_step_deg", 0.25)),
        fine_step_deg=float(spec.get("fine_step_deg", 0.05)),
        focus_z_index=int(auto_reg["large_focus_z_index"]),
    )

    prev_tx = float(row["small_to_large_top_left_x_px"])
    prev_ty = float(row["small_to_large_top_left_y_px"])
    prev_theta = float(row["small_to_large_theta_deg"])

    row["small_to_large_top_left_x_px"] = float(anchored["small_to_large_top_left_x_px"])
    row["small_to_large_top_left_y_px"] = float(anchored["small_to_large_top_left_y_px"])
    row["small_to_large_theta_deg"] = float(anchored["small_to_large_theta_deg"])
    row["small_to_large_best_z_index"] = int(anchored["small_to_large_best_z_index"])
    row["small_to_large_score"] = float(anchored["small_to_large_score"])
    row["small_to_large_alignment_method"] = str(anchored.get("small_to_large_alignment_method", "manual_top_left_anchor_rotation"))
    row["small_to_large_alignment_note"] = str(spec.get("note", ""))
    row["large_focus_z_index"] = int(anchored["large_focus_z_index"])
    row["large_focus_score"] = float(anchored["large_focus_score"])
    row["updated_at"] = datetime.now().isoformat(timespec="seconds")
    mwa.upsert_pair_transform_row(PAIR_TRANSFORM_TSV, row=row)

    override_rows.append({
        "canonical_position": str(position),
        "anchor_rule": str(spec.get("anchor_rule", "top_left")),
        "small_anchor_centroid_index": int(small_anchor_row["centroid_index"]),
        "large_anchor_centroid_index": int(large_anchor_row["centroid_index"]),
        "prev_top_left_x_px": prev_tx,
        "prev_top_left_y_px": prev_ty,
        "prev_theta_deg": prev_theta,
        "new_top_left_x_px": float(anchored["small_to_large_top_left_x_px"]),
        "new_top_left_y_px": float(anchored["small_to_large_top_left_y_px"]),
        "new_theta_deg": float(anchored["small_to_large_theta_deg"]),
        "new_score": float(anchored["small_to_large_score"]),
        "note": str(spec.get("note", "")),
    })

transforms_df = mwa.load_pair_transform_table(PAIR_TRANSFORM_TSV)
print("Manual small-to-large overrides applied:", len(override_rows))
if override_rows:
    override_df = pd.DataFrame(override_rows)
    display(override_df)
    display(
        transforms_df[
            transforms_df["canonical_position"].astype(str).isin([str(k) for k in MANUAL_SMALL_LARGE_OVERRIDES.keys()])
        ].copy()
    )


## Mapped Live Small Bead Wells

Regenerate the live small-image bead-centroid table from the finalized small-to-large transforms so downstream overlays and quantification use current geometry.

In [ ]:
LIVE_LARGE_CENTROIDS_TSV = ROOT / "results/annotations/well_centroids_large.tsv"

live_small_centroids_df = cmap.build_mapped_manual_centroid_table(
    large_centroid_path=LIVE_LARGE_CENTROIDS_TSV,
    transform_path=PAIR_TRANSFORM_TSV,
    out_path=LIVE_SMALL_MAPPED_CENTROIDS_TSV,
    cohort_ids=[COHORT_ID],
    require_ok_registration=True,
    verbose=True,
)

print("Saved mapped live small centroid table:", LIVE_SMALL_MAPPED_CENTROIDS_TSV)
print("Mapping status counts:")
display(live_small_centroids_df["mapping_status"].value_counts(dropna=False).to_frame("n"))
print("Beads inside live small FOV:", int(live_small_centroids_df["inside_small_fov"].fillna(False).astype(bool).sum()))
display(live_small_centroids_df.head())

## Review 2-6 Alignment Intermediates

This section is a dedicated manual-review block for `2-6`. It shows the small and large annotated well anchors, then compares the automatic BF-match geometry against the manual anchor override using the same large focus plane.


In [ ]:
# Render dedicated 2-6 review QC with anchor/well alignment intermediates.
import matplotlib.pyplot as plt
import numpy as np

manual_review_rows = []

for position, spec in MANUAL_SMALL_LARGE_OVERRIDES.items():
    pair_matches = [pp for pp in pairs if str(pp.canonical_position) == str(position)]
    if not pair_matches:
        print(f"Manual review skipped for {position}: pair not found in active pairs.")
        continue
    pair = pair_matches[0]

    review_row = transforms_df[transforms_df["canonical_position"].astype(str) == str(position)].copy()
    if review_row.empty:
        print(f"Manual review skipped for {position}: override transform row not found.")
        continue
    override_row = review_row.iloc[0].copy()

    large_loaded = mwa.load_large_registered_stack(pair=pair, root=ROOT)
    small_loaded = mwa.load_primary_image(
        mwa.ImageRecord(
            image_id=pair.image_id,
            cohort_id=pair.cohort_id,
            canonical_position=pair.canonical_position,
            file_path=pair.small_file_path,
        ),
        root=ROOT,
    )
    auto_reg = mwa.register_small_to_large(
        small_bf=small_loaded.bf,
        large_bf_stack=large_loaded.bf_stack,
        allow_rotation=ALLOW_ROTATION,
        max_abs_angle_deg=MAX_ABS_ANGLE_DEG,
        coarse_step_deg=COARSE_STEP_DEG,
        fine_step_deg=FINE_STEP_DEG,
        search_radius_px=SEARCH_RADIUS_PX,
    )
    auto_row = override_row.copy()
    for key, value in auto_reg.items():
        auto_row[key] = value
    auto_row["small_to_large_alignment_method"] = auto_reg.get("small_to_large_alignment_method", "bf_template_match")
    auto_row["small_to_large_alignment_note"] = auto_reg.get("small_to_large_alignment_note", "")

    small_sub = small_centroids_df[
        (small_centroids_df["cohort_id"].astype(str) == str(COHORT_ID))
        & (small_centroids_df["canonical_position"].astype(str) == str(position))
    ].copy().sort_values("centroid_index")
    large_sub = large_centroids_df[
        (large_centroids_df["cohort_id"].astype(str) == str(COHORT_ID))
        & (large_centroids_df["canonical_position"].astype(str) == str(position))
    ].copy().sort_values("centroid_index")

    small_anchor_row = mwa.select_centroid_by_spatial_rule(
        small_sub,
        rule=str(spec.get("anchor_rule", "top_left")),
    )
    large_anchor_row = mwa.select_centroid_by_spatial_rule(
        large_sub,
        rule=str(spec.get("anchor_rule", "top_left")),
    )
    small_anchor_xy = [(float(small_anchor_row["centroid_x_px"]), float(small_anchor_row["centroid_y_px"]))]
    large_anchor_xy = [(float(large_anchor_row["centroid_x_px"]), float(large_anchor_row["centroid_y_px"]))]

    def _anchor_residuals(transform_row, small_anchor_xy, large_anchor_xy):
        pred = [
            mwa.map_small_xy_to_large(
                x_small_px=float(x),
                y_small_px=float(y),
                small_shape_yx=(int(transform_row["small_shape_y_px"]), int(transform_row["small_shape_x_px"])),
                top_left_x_px=float(transform_row["small_to_large_top_left_x_px"]),
                top_left_y_px=float(transform_row["small_to_large_top_left_y_px"]),
                theta_deg=float(transform_row["small_to_large_theta_deg"]),
            )
            for x, y in small_anchor_xy
        ]
        pred = np.asarray(pred, dtype=np.float64)
        obs = np.asarray(large_anchor_xy, dtype=np.float64)
        res = np.sqrt(((pred - obs) ** 2).sum(axis=1))
        return float(np.mean(res)), float(np.max(res))

    auto_mean_res, auto_max_res = _anchor_residuals(auto_row, small_anchor_xy, large_anchor_xy)
    over_mean_res, over_max_res = _anchor_residuals(override_row, small_anchor_xy, large_anchor_xy)

    out_png = MANUAL_REVIEW_QC_DIR / f"{position}_small_large_manual_review_qc.png"
    all_small_labels = []
    for _, rr in small_sub.iterrows():
        lab = str(int(rr["centroid_index"]))
        if int(rr["centroid_index"]) == int(small_anchor_row["centroid_index"]):
            lab = ""
        all_small_labels.append(lab)
    all_large_labels = []
    for _, rr in large_sub.iterrows():
        lab = str(int(rr["centroid_index"]))
        if int(rr["centroid_index"]) == int(large_anchor_row["centroid_index"]):
            lab = ""
        all_large_labels.append(lab)

    fig = mwa.render_small_large_manual_review_qc(
        pair=pair,
        root=ROOT,
        automatic_transform_row=auto_row,
        override_transform_row=override_row,
        small_anchor_xy=small_anchor_xy,
        large_anchor_xy=large_anchor_xy,
        small_anchor_labels=["TL"],
        large_anchor_labels=["TL"],
        all_small_xy=small_sub[["centroid_x_px", "centroid_y_px"]].to_numpy(dtype=float),
        all_small_labels=all_small_labels,
        all_large_xy=large_sub[["centroid_x_px", "centroid_y_px"]].to_numpy(dtype=float),
        all_large_labels=all_large_labels,
        save_path=out_png,
    )
    plt.close(fig)

    manual_review_rows.append({
        "canonical_position": str(position),
        "anchor_rule": str(spec.get("anchor_rule", "top_left")),
        "small_anchor_centroid_index": int(small_anchor_row["centroid_index"]),
        "large_anchor_centroid_index": int(large_anchor_row["centroid_index"]),
        "auto_top_left_x_px": float(auto_row["small_to_large_top_left_x_px"]),
        "auto_top_left_y_px": float(auto_row["small_to_large_top_left_y_px"]),
        "auto_theta_deg": float(auto_row["small_to_large_theta_deg"]),
        "auto_score": float(auto_row["small_to_large_score"]),
        "auto_anchor_mean_residual_px": auto_mean_res,
        "auto_anchor_max_residual_px": auto_max_res,
        "override_top_left_x_px": float(override_row["small_to_large_top_left_x_px"]),
        "override_top_left_y_px": float(override_row["small_to_large_top_left_y_px"]),
        "override_theta_deg": float(override_row["small_to_large_theta_deg"]),
        "override_score": float(override_row["small_to_large_score"]),
        "override_anchor_mean_residual_px": over_mean_res,
        "override_anchor_max_residual_px": over_max_res,
        "saved_qc_path": str(out_png.relative_to(ROOT)),
        "review_note": str(spec.get("note", "")),
    })

manual_review_rows = sorted(manual_review_rows, key=lambda r: r["canonical_position"])
manual_review_df = pd.DataFrame(manual_review_rows)
MANUAL_REVIEW_QC_DIR.mkdir(parents=True, exist_ok=True)
manual_review_df.to_csv(MANUAL_REVIEW_QC_SUMMARY_TSV, sep="	", index=False)
print("Saved manual review summary:", MANUAL_REVIEW_QC_SUMMARY_TSV)
if not manual_review_df.empty:
    display(manual_review_df)
    for _, row in manual_review_df.iterrows():
        qc_path = ROOT / str(row["saved_qc_path"])
        display(Markdown(f"### {row['canonical_position']} review"))
        display(IPyImage(filename=str(qc_path), width=1350))
else:
    print("No manual review QC rows generated.")


## Save Alignment QC Outputs


In [ ]:
# Render QC PNGs for every pair and save a summary table.
qc_summary = mwa.render_alignment_qc_for_pairs(
    pairs=pairs,
    root=ROOT,
    transform_path=PAIR_TRANSFORM_TSV,
    qc_dir=QC_DIR,
    show_inline=SHOW_INLINE_QC,
    verbose=True,
)

QC_SUMMARY_TSV = QC_ROOT / "per_pair_summary.tsv"
QC_SUMMARY_TSV.parent.mkdir(parents=True, exist_ok=True)
qc_summary.to_csv(QC_SUMMARY_TSV, sep="	", index=False)

print("QC rows:", len(qc_summary))
print("Saved QC summary:", QC_SUMMARY_TSV)
if not qc_summary.empty:
    display(qc_summary)

missing_or_error = qc_summary[qc_summary["qc_status"] != "rendered"].copy() if not qc_summary.empty else qc_summary
if missing_or_error is not None and not missing_or_error.empty:
    print("QC issues:", len(missing_or_error))
    display(missing_or_error)
else:
    print("All pair QC panels rendered successfully.")


## Display Saved QC Panels


In [ ]:
# Display the saved QC PNGs inline from disk so the executed notebook is inspectable.
rendered_qc = qc_summary[qc_summary["qc_status"] == "rendered"].copy()
print(f"Embedding {len(rendered_qc)} saved QC panel(s) from disk into notebook output.")

for row in rendered_qc.itertuples(index=False):
    qc_rel = Path(str(row.saved_qc_path))
    qc_path = qc_rel if qc_rel.is_absolute() else (ROOT / qc_rel)
    if not qc_path.exists():
        raise FileNotFoundError(f"Missing saved QC PNG: {qc_path}")
    display(Markdown(f"### {row.canonical_position}"))
    display(IPyImage(filename=str(qc_path)))


## Validate Active Pair Coverage


In [ ]:
# Quick consistency check: all active pairs should have transform_status=ok and qc_status=rendered.
if len(pairs) == 0:
    print("No pairs to validate.")
else:
    wanted_large_files = {str(p.large_file_path) for p in pairs}
    active_transforms_df = transforms_df[transforms_df["large_file_path"].astype(str).isin(wanted_large_files)].copy()
    n_pairs = len(pairs)
    n_ok = int((active_transforms_df["registration_status"] == "ok").sum())
    n_qc = int((qc_summary["qc_status"] == "rendered").sum()) if len(qc_summary) > 0 else 0
    print(f"Pairs: {n_pairs} | transform ok: {n_ok} | QC rendered: {n_qc}")
    if (n_ok != n_pairs) or (n_qc != n_pairs):
        raise RuntimeError("Alignment validation incomplete. Review transform/QC tables above.")
